# Run the Wonderland Pipeline From Zero Setup to Result

**What this section does:** introduces the notebook, defines the practical end-to-end workflow that is actually implemented in this repository, and clarifies what counts as a result versus an evaluation.

This notebook is a **repo-specific onboarding and execution guide** for the current `rag-from-scratch` codebase. It is designed for a new engineer to run the checked-in workflow from local setup through profiling, routing, baseline evaluation, synthetic-data generation, SFT-data preparation, and optional LoRA submission packaging.

## What “final result” means in this repo
Because this repository does **not** yet include a checked-in trainer, vLLM inference runner, or hidden-test prediction script, the most honest “final result” you can produce today is:

1. a verified offline benchmark profile,
2. a verified router report,
3. a baseline evaluation report,
4. optional synthetic and SFT artifacts for future LoRA work, and
5. an optional packaged LoRA submission bundle **if you already have a Nemotron-compatible adapter directory**.

## What “evaluation” means in this repo
Evaluation in the current repo means running the **real checked-in offline evaluation path** in `scripts/eval_baselines.py`, which reports:
- overall exact-match accuracy,
- answer-format accuracy,
- per-family accuracy, and
- random-vs-structure-aware split comparisons.

> **Important assumption:** the visible `test.csv` is only a smoke-test sample and overlaps with `train.csv`, so it is **not** a trustworthy validation set for experiment selection.


## 1. Environment and prerequisites

**What this section does:** explains the assumptions needed to run the notebook from top to bottom and highlights which steps are lightweight versus optional.

### Python assumptions
- Recommended: **Python 3.10+**.
- The checked-in project scripts use the **Python standard library only**.
- To run this file as a notebook, you also need a Jupyter environment.

### Minimal environment setup
```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip notebook
```

### GPU / CUDA assumptions
- **No GPU is required** for the implemented workflow in this notebook.
- GPU, CUDA, PEFT, PyTorch, and vLLM only become relevant when you train or serve a real Nemotron LoRA outside the currently checked-in code.

### Working-directory assumption
Run Jupyter from the **repository root** so relative paths such as `train.csv` and `scripts/profile_dataset.py` resolve correctly.

### Required input files
This notebook expects these files to exist before execution:
- `train.csv`
- `test.csv`
- `scripts/profile_dataset.py`
- `scripts/build_router.py`
- `scripts/eval_baselines.py`
- `scripts/generate_synthetic.py`
- `scripts/prepare_sft_data.py`
- `scripts/package_lora_submission.py`


In [ ]:
from __future__ import annotations

import csv
import json
import os
import platform
import re
import subprocess
import sys
from itertools import islice
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path.cwd().resolve()
NOTEBOOK_ROOT = REPO_ROOT / "notebooks"
ARTIFACT_ROOT = REPO_ROOT / "artifacts"

print(f"Repository root: {REPO_ROOT}")
print(f"Python version: {platform.python_version()}")
print(f"Python executable: {sys.executable}")

required_paths = [
    REPO_ROOT / "train.csv",
    REPO_ROOT / "test.csv",
    REPO_ROOT / "scripts" / "profile_dataset.py",
    REPO_ROOT / "scripts" / "build_router.py",
    REPO_ROOT / "scripts" / "eval_baselines.py",
    REPO_ROOT / "scripts" / "generate_synthetic.py",
    REPO_ROOT / "scripts" / "prepare_sft_data.py",
    REPO_ROOT / "scripts" / "package_lora_submission.py",
]

missing = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required repo paths: {missing}")

print("All required repo inputs are present.")
print("\nWhat to check: confirm the notebook is running from the repository root and that no required files are missing.")


## 2. Repository discovery

**What this section does:** shows the real repository shape, identifies which directories currently exist, and maps the notebook to the checked-in scripts instead of reimplementing logic.

This notebook intentionally reuses the existing CLI scripts wherever possible:
- `scripts/profile_dataset.py` for benchmark profiling,
- `scripts/build_router.py` for routing validation,
- `scripts/eval_baselines.py` for the real offline evaluation path,
- `scripts/generate_synthetic.py` for approved synthetic data,
- `scripts/prepare_sft_data.py` for chat-style SFT datasets,
- `scripts/package_lora_submission.py` for packaging validation.

> **Repo-specific note:** the current repository does **not** contain checked-in `src/`, `configs/`, `experiments/`, or `outputs/` workflow code. The implemented business logic lives in `scripts/` and the benchmark analysis lives in `docs/`.


In [ ]:
interesting_dirs = ["docs", "scripts", "src", "configs", "experiments", "outputs", "notebooks"]
for dirname in interesting_dirs:
    path = REPO_ROOT / dirname
    print(f"\n== {dirname}/ exists={path.exists()} ==")
    if path.exists():
        files = sorted(p.relative_to(REPO_ROOT).as_posix() for p in path.rglob("*") if p.is_file())
        for item in files[:50]:
            print(f"  {item}")
        if len(files) > 50:
            print(f"  ... ({len(files) - 50} more files)")

print("\nCore docs reused by this notebook:")
for doc_name in [
    "docs/dataset_forensics.md",
    "docs/task_taxonomy.md",
    "docs/system_architecture.md",
    "docs/eval_plan.md",
    "docs/lora_plan.md",
    "docs/run_from_zero_to_result.md",
]:
    print(f"  - {doc_name}: {'present' if (REPO_ROOT / doc_name).exists() else 'missing'}")

print("\nWhat to check: the repo should show script-centric workflow files under scripts/ and benchmark notes under docs/.")


In [ ]:
def run_command(command: list[str], *, cwd: Path = REPO_ROOT, env: dict[str, str] | None = None, check: bool = True) -> str:
    """Run a repo command, stream stdout/stderr after completion, and return stdout."""
    print("$", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with code {completed.returncode}: {' '.join(command)}")
    return completed.stdout


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


print("Helper utilities loaded.")
print("\nWhat to check: later cells should call run_command(...) instead of duplicating CLI logic in the notebook.")


## 3. Quick start path

**What this section does:** runs the shortest path to a first working result using the existing scripts exactly as they are implemented today.

The fastest honest path to a first result is:
1. profile the dataset,
2. validate the router,
3. run the offline baseline evaluation.

Expected outputs:
- terminal summary from `profile_dataset.py`,
- router metrics from `build_router.py`,
- baseline accuracy report from `eval_baselines.py`.


In [ ]:
quick_profile_output = run_command([sys.executable, "scripts/profile_dataset.py"])
quick_router_output = run_command([sys.executable, "scripts/build_router.py", "--skip-demo"])
quick_eval_output = run_command([sys.executable, "scripts/eval_baselines.py", "--baseline", "solver_lite"])

print("Quick-start path finished.")
print("\nWhat to check: profile output should show 9,500 train rows and the router/baseline commands should complete without errors.")


## 4. Data inspection

**What this section does:** inspects the raw CSV inputs directly before any heavier workflow step, matching the repo instructions in `AGENTS.md`.


In [ ]:
for csv_name in ["train.csv", "test.csv"]:
    path = REPO_ROOT / csv_name
    print(f"\n== Preview: {csv_name} ==")
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in islice(reader, 2):
            preview = {
                key: (value[:160] + "..." if isinstance(value, str) and len(value) > 160 else value)
                for key, value in row.items()
            }
            pprint(preview)

print("\nWhat to check: confirm the benchmark really is prompt/answer data and that test.csv has no answer column.")


## 5. End-to-end pipeline

**What this section does:** runs the full practical workflow supported by the checked-in repository from dataset profiling through optional packaging.

The notebook stays aligned to the actual codebase:
- it uses subprocess calls to the checked-in scripts,
- it avoids reimplementing solver logic inside notebook cells,
- and it clearly labels optional sections where the repo depends on external training or adapter artifacts.


### 5.1 Dataset profiling

**What this section does:** materializes a machine-readable dataset profile that can be saved and compared across future runs.


In [ ]:
ARTIFACT_ROOT.mkdir(exist_ok=True)
profile_json_path = ARTIFACT_ROOT / "profile_dataset_notebook.json"
profile_stdout = run_command([sys.executable, "scripts/profile_dataset.py", "--json"])
profile_json_path.write_text(profile_stdout, encoding="utf-8")
profile = json.loads(profile_stdout)

print(f"Saved machine-readable profile to: {profile_json_path.relative_to(REPO_ROOT)}")
print("Train rows:", profile["overall"]["train_rows"])
print("Visible test rows:", profile["overall"]["test_rows"])
print("Exact visible test/train overlap:", profile["overall"]["exact_test_prompt_overlap_with_train"])
print("Families:", sorted(profile["families"].keys()))
print("\nWhat to check: the saved JSON should exist under artifacts/ and confirm that visible test rows overlap with train.")


### 5.2 Router validation

**What this section does:** runs the real family-router evaluation path and preserves the text output for later comparison.


In [ ]:
router_report_path = ARTIFACT_ROOT / "router_eval_notebook.txt"
router_stdout = run_command([sys.executable, "scripts/build_router.py", "--skip-demo"])
router_report_path.write_text(router_stdout, encoding="utf-8")

print(f"Saved router report to: {router_report_path.relative_to(REPO_ROOT)}")
print("\nWhat to check: top-level routing accuracy should be 1.0000 on the visible train split.")


### 5.3 Offline evaluation

**What this section does:** runs the repository's real evaluation script and extracts the most important metrics for quick review.


In [ ]:
eval_report_path = ARTIFACT_ROOT / "baseline_eval_notebook.txt"
eval_stdout = run_command([sys.executable, "scripts/eval_baselines.py", "--baseline", "solver_lite"])
eval_report_path.write_text(eval_stdout, encoding="utf-8")

parsed_eval = {}
current_split = None
in_family_block = False
for raw_line in eval_stdout.splitlines():
    line = raw_line.rstrip()
    stripped = line.strip()
    if stripped.endswith(":") and stripped[:-1] in {"stratified_random_hash", "structure_aware_hash"}:
        current_split = stripped[:-1]
        parsed_eval[current_split] = {"per_family_accuracy": {}}
        in_family_block = False
        continue
    if current_split is None:
        continue
    if stripped.startswith("overall_accuracy="):
        parsed_eval[current_split]["overall_accuracy"] = float(stripped.split("=", 1)[1].split()[0])
        continue
    if stripped.startswith("answer_format_accuracy="):
        parsed_eval[current_split]["answer_format_accuracy"] = float(stripped.split("=", 1)[1].split()[0])
        continue
    if stripped == "per_family_accuracy:":
        in_family_block = True
        continue
    if in_family_block and stripped.startswith("-"):
        family, score = stripped[1:].split()
        parsed_eval[current_split]["per_family_accuracy"][family] = float(score)
        continue
    if in_family_block and stripped and not stripped.startswith("-"):
        in_family_block = False

print(json.dumps(parsed_eval, indent=2, sort_keys=True))
print(f"Saved evaluation report to: {eval_report_path.relative_to(REPO_ROOT)}")
print("\nWhat to check: overall_accuracy should be about 0.4336 and answer_format_accuracy should be 1.0000 for solver_lite.")


### 5.4 Synthetic data generation

**What this section does:** generates the repo's approved lightweight synthetic curriculum using the checked-in generator.

This is a **lightweight demo path** because the generator is already constrained to the approved families:
- `roman_numeral`
- `unit_conversion`
- `gravity`


In [ ]:
synthetic_output = ARTIFACT_ROOT / "synthetic" / "synthetic_train_notebook_mvp.csv"
synthetic_output.parent.mkdir(parents=True, exist_ok=True)
run_command([
    sys.executable,
    "scripts/generate_synthetic.py",
    "--preset", "mvp",
    "--output", str(synthetic_output),
    "--seed", "7",
])

with synthetic_output.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    synthetic_rows = list(reader)

print(f"Synthetic rows written: {len(synthetic_rows)}")
print("Families in synthetic data:", sorted({row['family'] for row in synthetic_rows}))
print("First synthetic row id:", synthetic_rows[0][next(iter(synthetic_rows[0].keys()))])
print("\nWhat to check: the output CSV should exist under artifacts/synthetic/ and only contain the approved families.")


### 5.5 SFT data preparation

**What this section does:** converts real and optional synthetic rows into the repo's chat-style JSONL format for future LoRA experiments.

This is still part of the checked-in workflow even though the repo does **not** yet include a training script.


In [ ]:
sft_output_dir = ARTIFACT_ROOT / "sft_notebook"
run_command([
    sys.executable,
    "scripts/prepare_sft_data.py",
    "--synthetic-path", str(synthetic_output),
    "--output-dir", str(sft_output_dir),
    "--recipe", "mixed_sft",
    "--box-style", "boxed",
    "--max-synthetic-per-family", "100",
])

for path in sorted(sft_output_dir.glob("*.jsonl")):
    line_count = sum(1 for _ in path.open("r", encoding="utf-8"))
    print(f"{path.relative_to(REPO_ROOT)} -> {line_count} rows")

sample_jsonl = next(iter(sorted(sft_output_dir.glob("*.jsonl"))))
with sample_jsonl.open("r", encoding="utf-8") as handle:
    first_record = json.loads(handle.readline())
print("\nSample SFT record keys:", first_record.keys())
print("Sample record metadata:")
pprint(first_record["metadata"])
print("\nWhat to check: train/dev JSONL files should be created and assistant targets should use boxed answers when box-style=boxed.")


### 5.6 Inference / result generation status

**What this section does:** makes the current repository boundary explicit so a new engineer does not assume missing functionality exists.

> **Current repo gap:** there is no checked-in training script, no vLLM inference launcher, and no script that reads `test.csv` and emits final hidden-test predictions. In this repo version, the practical top-to-bottom path ends at offline evaluation, training-data preparation, and optional submission packaging.

The "result generation" step that is currently implemented is therefore:
- offline baseline result generation via `scripts/eval_baselines.py`, and
- artifact generation for later LoRA work via `scripts/generate_synthetic.py` and `scripts/prepare_sft_data.py`.


### 5.7 Optional submission packaging

**What this section does:** demonstrates the real packaging path. If you already have an adapter directory, point the notebook to it. Otherwise, the notebook creates a minimal **config-only dry-run adapter** so the packaging workflow can still be exercised safely.


In [ ]:
user_adapter_dir = os.environ.get("NOTEBOOK_ADAPTER_DIR", "").strip()
if user_adapter_dir:
    adapter_dir = Path(user_adapter_dir).expanduser().resolve()
    allow_missing = False
    print(f"Using existing adapter directory: {adapter_dir}")
else:
    adapter_dir = ARTIFACT_ROOT / "demo_adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    demo_config = {
        "base_model_name_or_path": "nvidia/Nemotron-3-Nano-30B-Instruct",
        "peft_type": "LORA",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.0,
    }
    (adapter_dir / "adapter_config.json").write_text(json.dumps(demo_config, indent=2) + "\n", encoding="utf-8")
    allow_missing = True
    print(f"Created config-only demo adapter at: {adapter_dir.relative_to(REPO_ROOT)}")

submission_output_dir = ARTIFACT_ROOT / "submission_notebook"
command = [
    sys.executable,
    "scripts/package_lora_submission.py",
    "--adapter-dir", str(adapter_dir),
    "--output-dir", str(submission_output_dir),
    "--submission-name", "notebook_demo_submission",
]
if allow_missing:
    command.append("--allow-missing-weight-file")

package_stdout = run_command(command)
package_info = json.loads(package_stdout)
pprint(package_info)

manifest_path = Path(package_info["submission_dir"]) / "manifest.json"
manifest = read_json(manifest_path)
print("\nManifest keys:", sorted(manifest.keys()))
print("\nWhat to check: packaging should produce a submission directory, a tar.gz archive, and a manifest.json file.")


## 6. Evaluation guidance

**What this section does:** explains how to interpret the repo's real evaluation output and how to compare experiments fairly.

### What the reported metrics mean
- **overall accuracy**: exact-match accuracy across all evaluated rows.
- **answer-format accuracy**: whether the prediction matches the expected output schema, even when the answer itself is wrong.
- **per-family accuracy**: family-by-family exact-match performance.
- **random vs structure-aware comparison**: whether the method generalizes across latent structural buckets instead of only surface-near rows.

### Fair experiment comparison rules in this repo
1. Compare runs on the **same baseline script and split settings**.
2. Do **not** use the visible `test.csv` as a model-selection target.
3. Keep validation **real-only** when synthetic rows are introduced for training.
4. Compare both:
   - exact match, and
   - answer-format accuracy.
5. Review per-family movement, especially for the unsolved families:
   - `bit_transform`
   - `text_cipher`
   - `equation_transform`

### Warning about misleading evaluation
- The visible `test.csv` is fully overlapped with train and should only be treated as a smoke test.
- Strong gains on easy deterministic families can hide the fact that hard families remain unsolved.
- Format-only gains can look deceptively good if they do not improve exact match.
- Structure-aware comparisons are more informative than random splits when hidden-test drift is possible.


In [ ]:
print("Parsed evaluation summary:")
for split_name, metrics in parsed_eval.items():
    print(f"\n[{split_name}]")
    print(f"  overall_accuracy       = {metrics['overall_accuracy']:.4f}")
    print(f"  answer_format_accuracy = {metrics['answer_format_accuracy']:.4f}")
    print("  per_family_accuracy    =")
    for family, score in metrics["per_family_accuracy"].items():
        print(f"    - {family:18s} {score:.4f}")

hard_families = ["bit_transform", "text_cipher", "equation_transform"]
print("\nHard-family snapshot:")
for family in hard_families:
    score = parsed_eval["stratified_random_hash"]["per_family_accuracy"][family]
    print(f"  {family:18s} {score:.4f}")

print("\nWhat to check: easy-family accuracy should dominate today, while bit_transform/text_cipher/equation_transform remain the main opportunity areas.")


## 7. Results inspection and validation

**What this section does:** shows where outputs were written, how to inspect them, and how to tell whether the run actually succeeded.


In [ ]:
expected_artifacts = [
    profile_json_path,
    router_report_path,
    eval_report_path,
    synthetic_output,
    *sorted(sft_output_dir.glob("*.jsonl")),
    submission_output_dir / "notebook_demo_submission",
    submission_output_dir / "notebook_demo_submission.tar.gz",
]

print("Generated artifacts:")
for artifact in expected_artifacts:
    print(f"  - {artifact.relative_to(REPO_ROOT)} :: exists={artifact.exists()}")

print("\nCommon failure patterns to watch for:")
print("  1. Running the notebook from the wrong working directory.")
print("  2. Assuming test.csv is a true validation set.")
print("  3. Expecting a checked-in training or inference runner that does not exist yet.")
print("  4. Forgetting that packaging requires a real adapter directory unless dry-run mode is used.")
print("  5. Comparing experiments without keeping the evaluation split strategy fixed.")

print("\nWhat to check: every expected artifact above should exist after a successful top-to-bottom run.")


## 8. Reproducibility and next steps

**What this section does:** records what should be saved from a run and suggests the highest-value next experiment path based on the actual repository state.

### Save these artifacts
- `artifacts/profile_dataset_notebook.json`
- `artifacts/router_eval_notebook.txt`
- `artifacts/baseline_eval_notebook.txt`
- `artifacts/synthetic/synthetic_train_notebook_mvp.csv`
- `artifacts/sft_notebook/*.jsonl`
- `artifacts/submission_notebook/notebook_demo_submission*`

### What to pin
- Python version
- notebook execution order
- synthetic generation seed
- evaluation baseline name
- any future training config, LoRA rank, and adapter base model once training exists

### How to rerun fairly
- rerun the notebook from the repo root,
- keep the same seeds and CLI flags,
- avoid changing evaluation settings between comparison runs,
- compare artifacts and metrics side by side.

### Suggested next experiment path
Based on the current repo implementation, the next highest-value steps are:
1. build stronger solver coverage for `bit_transform`,
2. reverse-engineer the `text_cipher` family,
3. deepen `equation_transform` taxonomy and solver search,
4. only then test minimal format-focused LoRA training using the prepared SFT data.


## Definition of Success

**What this section does:** summarizes the expected outputs, the key checks that indicate success, and the most common reasons a top-to-bottom run may fail.

A notebook run is successful if it produces the following practical outputs:
- a saved dataset profile JSON,
- a saved router evaluation report,
- a saved offline baseline evaluation report,
- a synthetic CSV for the approved families,
- train/dev SFT JSONL files,
- and a submission package directory plus archive when packaging is exercised.

### Expected checks
- dataset profiling confirms the visible `test.csv` is only a smoke-test sample,
- router validation completes successfully,
- baseline evaluation reports overall accuracy and per-family accuracy,
- SFT preparation writes JSONL files with boxed-answer targets when requested,
- packaging writes `manifest.json` and an archive.

### Common reasons the run may fail
- the notebook is launched outside the repository root,
- required CSV or script files are missing,
- a user expects missing training/inference functionality that is not checked in,
- a real adapter directory is not available and dry-run packaging is not used,
- or experiment comparisons are made against the leaked visible `test.csv` instead of the real offline evaluation path.
